# Libraries

In [ ]:
# General
import os # For working with directories
import numpy as np # For math
#import pandas as pd
from tensorflow.config import list_physical_devices # To see if we're using GPU
from tkinter.filedialog import askopenfilenames, askdirectory # File dialog
from tkinter import Tk # File dialog
from numba import cuda # Used to release the GPU memory, if needed
import random

# Preprocessing
from OrganizeData import organize_downloaded_images, check_rotated_images,\
fix_wrong_labels, copy_annotations, train_val_test_split, save_annotations_df, train_val_test_split_balanced

# Images
import cv2 # For adding polygons to images
import matplotlib.pyplot as plt # For visualization
from matplotlib.colors import LinearSegmentedColormap # Consistent color map for all segmentations
from matplotlib.colors import to_rgb # Translate text to RGB

# Reading JSON files
# Now happens from copy_annotations function at OrganizeData
from json_data_class import JSON_data

from sklearn.metrics import classification_report, confusion_matrix

# Image generator: we'll have a lot of images, instead of reading every one to memory, we'll feed them batch by batch
from image_generator_custom import data_gen

# Unet (Deep learning) Model:
from MultiClassUnet import multi_unet_model
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam

In [ ]:
# Making sure we're using GPU
print(list_physical_devices())
print("Num GPUs Available: ", len(list_physical_devices('GPU')))

In [ ]:
import GPUtil

gpus = GPUtil.getGPUs()
gpu_names = [gpu.name for gpu in gpus]

print(', '.join(gpu_names))

In [ ]:
os.getcwd()

In [ ]:
if os.name =='nt': # Windows
    prefix_os_c = 'C:/'
    prefix_os_d = 'D:/'
elif os.name =='posix': # WSL/Linux
    prefix_os_c = '/mnt/c'
    prefix_os_d = '/mnt/d'
os.chdir(os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python'))

# Organize data

In [ ]:
# whether we want to use Segmentation Models library, if we need to add a 3rd dimension in the data_gen function
use_sm = True

# If we need to organize the data, use this
organize_data = False

# If we want to see specific annotations or not
specific_annot = False

# If we need to organize the specific dataset (i.e, we change the labels)
organize_data_specific = False

# Dictionary of labels

In [ ]:
# Dictionary of labels
labels = []
txt = 'My_labels.txt'

with open(txt) as labels_file:

    for line in labels_file.readlines():
        labels.append(line.replace('\n','')) 

labels_dict = {}
for indx, label in enumerate(labels):
    indx = indx+1 # Background is the pixel number 0
    labels_dict[label] = (indx,indx,indx)
    
print(labels_dict)

In [ ]:
labels

In [ ]:
def choose_dir(text = '') -> str:
    '''
    A function that helps the user to choose a folder
    Input: text:str - Choose which part of the data is needed:
           'train' for training, 'val' for validation, 'test' for test.
           The default value is 'train'.
           
    Output: The path to the desired data
    '''
    root = Tk()
    root.withdraw()
    # To show the dialog at the front, we need both:
    # 1: root.wm_attributes('-topmost', 1)
    root.wm_attributes('-topmost', 1)
    # 2: parent = root 
    path = askdirectory(parent=root, title= text)
    return(path)

In [ ]:
print(os.name)

In [ ]:
#main_folder_path = choose_dir(text = 'Choose the main folder, where all the subfolders are')
#main_folder_path = r'D:\Aviel\לימודים\תואר שני - הנדסת חשמל בר אילן\תזה\2) Final thesis\Dataset - Dental Images'
#main_folder_path = 'Aviel/לימודים/תואר שני - הנדסת חשמל בר אילן/תזה/2) Final thesis/Dataset - Dental Images'
main_folder_path = os.path.join(prefix_os_d,'Aviel/לימודים/תואר שני - הנדסת חשמל בר אילן/תזה/2) Final thesis/Dataset - Dental Images')

all_pics_path = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/All pics')
bitewing_path = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/Bitewing')
periapical_path = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/Periapical')


train_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/train/train_pics')
val_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/val/val_pics')
test_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/test/test_pics')

src_path = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python')

In [ ]:
if organize_data: # Need to run only once, when there is a change in the dataset (i.e, new annotations)    
    
    
    #fix_wrong_labels(annotations_path = main_folder_path)
    
    suffix = ['Annotations','Not relevant','No pathology']
    #organize_downloaded_images(main_folder_path = main_folder_path, suffix_sub_folder = suffix)
    
    suffix = ['Periapical','Bitewing']
    #copy_annotations(src_path = main_folder_path, dest_path = all_pics_path, bite_peri_split=True,
    #dest_path_bitewing = bitewing_path, dest_path_periapical = periapical_path, suffix = suffix, labels_dict = labels_dict)
    
    #check_rotated_images(main_folder_path = all_pics_path)
        
    #check_rotated_images(main_folder_path = bitewing_path)
    #check_rotated_images(main_folder_path = periapical_path)

    '''
    train_val_test_split(src_path = all_pics_path, dest_path_train = train_path_pics,
                         dest_path_val = val_path_pics, dest_path_test = test_path_pics, labels = labels)   
    '''

    #train_val_test_split_balanced(src_path = all_pics_path, dest_path_train = train_path_pics,
    #                     dest_path_val = val_path_pics, dest_path_test = test_path_pics, labels_org = labels,
    #                              train_percent=0.7, test_percent = 0.15, seed = 101)         
   
    
    save_annotations_df(df_path = src_path, train_path = train_path_pics, val_path = val_path_pics,
                        test_path = test_path_pics, labels = labels, df_name = 'all_annotations')       

In [ ]:
# For specific annotations
if specific_annot:
    
    labels = []

    os.chdir(src_path)
    txt = 'My_labels_specific.txt'
    with open(txt) as labels_file:
        for line in labels_file.readlines():
            labels.append(line.replace('\n','')) 
            
    labels_dict = {}
    for indx, label in enumerate(labels):
        indx = indx+1 # Background is the pixel number 0
        labels_dict[label] = (indx,indx,indx)  

    train_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/train_specific/train_pics')
    val_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/train_specific/val_pics')
    test_path_pics = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/train_specific/test_pics')
    
    #train_path_pics = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\train_specific\train_pics'
    #val_path_pics = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\val_specific\val_pics'
    #test_path_pics = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\test_specific\test_pics'
    
    if not(os.path.exists(train_path_pics)):
        os.makedirs(train_path_pics)
    if not(os.path.exists(val_path_pics)):
        os.makedirs(val_path_pics)
    if not(os.path.exists(test_path_pics)):
        os.makedirs(test_path_pics)
    
if specific_annot and organize_data_specific:
    
    #train_val_test_split(src_path = all_pics_path, dest_path_train = train_path_pics,
    #                     dest_path_val = val_path_pics, dest_path_test = test_path_pics, labels = labels)

    train_val_test_split_balanced(src_path = all_pics_path, dest_path_train = train_path_pics,
                         dest_path_val = val_path_pics, dest_path_test = test_path_pics, labels_org = labels,
                                  train_percent=0.7, test_percent = 0.15, seed = 101)        
     
    save_annotations_df(df_path = src_path, train_path = train_path_pics, val_path = val_path_pics,
                    test_path = test_path_pics, labels = labels, df_name = 'specific_annotations') 
    

# Extrect the masks from the JSON files using a custom made class

## Path for windows

In [ ]:
train_path_mask = os.path.join(train_path_pics,r'../train_masks')
train_path_bbox = os.path.join(train_path_pics,r'../train_bbox')

if not(os.path.exists(train_path_mask)):
    os.makedirs(train_path_mask)
if not(os.path.exists(train_path_bbox)):
    os.makedirs(train_path_bbox)


#val_path_pics = choose_dir(text='Choose the folder where the **val** images are located''')
#val_path_pics = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\val\val_pics'


val_path_mask = os.path.join(val_path_pics,r'../val_masks')
val_path_bbox = os.path.join(val_path_pics,r'../val_bbox')

if not(os.path.exists(val_path_mask)):
    os.makedirs(val_path_mask)
if not(os.path.exists(val_path_bbox)):
    os.makedirs(val_path_bbox)

#test_path_pics = choose_dir(text='Choose the folder where the **test** images are located''')
#test_path_pics = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\test\test_pics'

test_path_mask = os.path.join(test_path_pics,r'../test_masks')
test_path_bbox = os.path.join(test_path_pics,r'../test_bbox')

if not(os.path.exists(test_path_mask)):
    os.makedirs(test_path_mask)
if not(os.path.exists(test_path_bbox)):
    os.makedirs(test_path_bbox)
    

# Hyper-parameters

In [ ]:
# The amount of training epochs
epochs = 200

# How many images per batch
#batch_size = 4
#batch_size = 8
batch_size = 16

# Learning rate
#lr = 5e-4 #0.0005 Works as of 14/02/2024
# lr = 1e-7 # Works best as of 15/04/2025
#lr = 1e-8 # Works best as of 23/05/2025
lr = 1e-4 # Works best as of 04/12/2025



# Focal Loss:
# lr that don't work well: 1e-5, 1e-2, 1e-8
# Good lr: 1e-3, 1e-4, 5e-3

# Dice Loss:
# lr that don't work well: 
# Good lr: 

# The random seed
seed = 101
random.seed(seed)
np.random.seed(seed)

# The size of the images: rows,cols,color_channel (grayscale)
# The bitweing images's shape is: [,,1]
#image_size = [320,320,1]
# Our images from the IDF's database are around 800x620 pixles, so we'll square them to the nearest power of 2: 512
#image_size = [512,512,1]

image_size = [512,512,3]

if not use_sm:
    os.environ['TF_DETERMINISTIC_OPS'] = '1'  # tensorrflow gpu fix seed, please `pip install tensorflow-determinism` first
    tf.random.set_seed(seed) # Random seed for tensorflow
    tf.config.experimental.enable_op_determinism()

# Used for ModelCheckpoint
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Optional: Limit the GPU use
# Option 1: More flexible, but you don't know how much memory you'll need
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only use the first GPU
  try:
    tf.config.set_visible_devices(gpus[0], 'GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPU")
  except RuntimeError as e:
    # Visible devices must be set before GPUs have been initialized
    print(e)

# Option 2: A bit dangerous, you don't want to limit too much
'''
gpu_options = tf.compat.v1.GPUOptions(per_process_gpu_memory_fraction=0.666)
sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(gpu_options=gpu_options))
'''

# How many classes we have *In total*
#n_classes = 11
n_classes = len(labels) + 1

num_images_train = len([file for file in os.listdir(train_path_pics) if file.endswith(".jpg") or file.endswith(".png") or file.endswith(".tif")]) #List of training images)

num_images_val = len([file for file in os.listdir(val_path_pics) if file.endswith(".jpg") or file.endswith(".png") or file.endswith(".tif")]) #List of val images)

num_images_test = len([file for file in os.listdir(test_path_pics) if file.endswith(".jpg") or file.endswith(".png") or file.endswith(".tif")]) #List of val images)

In [ ]:
print(f'There are {num_images_train} training images')
print(f'There are {num_images_val} validation images')
print(f'There are {num_images_test} test images')

In [ ]:
if organize_data or (organize_data_specific and specific_annot):
    # Get the data from the JSON files: For train, val and test
    train_json = JSON_data(color_dict = labels_dict)
    train_json.save_annotatios_mask(train_path_pics, train_path_mask)
    #train_json.save_bbox(train_path_pics, train_path_bbox)

In [ ]:
if organize_data or (organize_data_specific and specific_annot):
    val_json = JSON_data(color_dict = labels_dict)
    val_json.save_annotatios_mask(val_path_pics, val_path_mask)
    #val_json.save_bbox(val_path_pics, val_path_bbox)

In [ ]:
if organize_data or (organize_data_specific and specific_annot):
    test_json = JSON_data(color_dict = labels_dict)
    test_json.save_annotatios_mask(test_path_pics, test_path_mask)
    #test_json.save_bbox(test_path_pics, test_path_bbox)

# Function - plot masks (segmentations) with colors

In [ ]:
def imshow_mask_or_overlay(mask:np.ndarray, labels_list:list, orig_img:np.ndarray=None,
                           title:str=None, fig_size=(12,12), mask_compare:np.ndarray = None):
    '''
    Input - 
            mask: The corresponding mask, shape of [row, cols, number_of_classes]
            labels_list: A list of all the labels
            orig_img: The original image [row, cols, n_channels (1 or 3)], if provided will overlay the mask on it
            title: The title of the image
            fig_size: The size of the output figure
            mask_compare : A mask that can be used to compare (for example, ground truth vs prediction)

    Output - An image with colored labels
    '''
    
    labels = ['Background'] + labels_list
    
    # For pet dataset, use only labels_list
    #print('For teeth segmentation, add background!')
    
    #labels = labels_list
    
    my_colors = ['k', 'b', 'r', 'g', 'y', 'c', 'm', 'tab:orange', 'royalblue', 'tab:brown', 'fuchsia'] # colors for 0, 1, 2
            
    # Create a color map from our chosen colors, maps the first color to pixel value 0, the second to pixel value 1...
    cmap = LinearSegmentedColormap.from_list('', my_colors, len(my_colors));

    if len(mask.shape)>2: # If we have logits
        mask = tf.math.argmax(mask,axis=-1)
    
    if mask_compare is not None: # If we want to compare, then make a 1 on 2 subplot

        if len(mask_compare.shape)>2: # If we have logits
            mask_compare = tf.math.argmax(mask_compare,axis=-1)

        fig, ax = plt.subplots(nrows = 1,ncols = 2,figsize = fig_size); # We don't actually need "fig"

        # Plot the legend:
        # Since imshow can't really show a legend, we first plot something, and then overide it, while maintaining the legend
        for indx, color in enumerate(my_colors[:len(labels)]):
            #ax[0].plot(2, 50, "-", color=color, label=labels[indx]);
            ax[1].plot(2, 50, "-", color=color, label=labels[indx]);

        if title is not None:
             #ax[0].set_title(title + ': 1');
             #ax[1].set_title(title + ': 2');
             ax[0].set_title(title);
             ax[1].set_title(title);



        if orig_img is None: # If we just want to plot the mask, no need for original image
        # Plot the image
            ax[0].imshow(mask,cmap=cmap, vmin=0, vmax=len(my_colors) - 1);          
            ax[0].legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0));
            ax[1].imshow(mask_compare,cmap=cmap, vmin=0, vmax=len(my_colors) - 1);          
            ax[1].legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0));
        
        else: # If we want to draw the mask on top of the original image
            ## Find and draw the edges
            # Make a copy to show the segmented image
            
            mask_copy = np.copy(mask)
            mask_copy_compare = np.copy(mask_compare)
    
            # Original image, copied
            if np.max(orig_img)>1.0:
                orig_img = orig_img/255
            new_img = np.copy(np.uint8(orig_img*255)) # uint 8 for the erode function
            # One less dimension (no need for a 1 dimension in the end)
            new_img = new_img[...,0]
            # Stack the same picture to get 3 color channels
            new_img_3d = np.stack((new_img,new_img,new_img),axis=-1)
            new_img_3d_compare = np.stack((new_img,new_img,new_img),axis=-1)
    
            # Find the unique classes in the mask
            unique_val = np.delete(np.unique(mask_copy),[0]) # Delete the 0th index, we don't want to segment the background
            unique_val_compare = np.delete(np.unique(mask_copy_compare),[0]) # Delete the 0th index, we don't want to segment the background         

            ### First mask
            for label_indx in unique_val: # Find the mask for each unique label
                # Create a new mask, one for each label
                new_mask = np.uint8((mask_copy==label_indx)) # uint8 for the erode function
                # Find the edges of the mask
                kernel = np.ones((3, 3), np.uint8)
                img_erod = cv2.erode(new_mask, kernel)
                mask_edge  = new_mask - img_erod
    
    
                # In order to draw the image, find each contour's coordinates.
                contours, _ = cv2.findContours(mask_edge, cv2.RETR_CCOMP , cv2.CHAIN_APPROX_SIMPLE)
    
                for cont_indx in range(len(contours)):
                    vertices = np.array(contours[cont_indx],np.int32) # cv2 requires int32 points
                    pts = vertices.reshape(-1,1,2) # cv2 requires this format
    
    
                    cv2.polylines(new_img_3d,[pts],isClosed=True,color=np.multiply(to_rgb(my_colors[label_indx]),255),thickness=1) # Adds the polygon to the original image
             ### Second mask
            for label_indx in unique_val_compare: # Find the mask for each unique label
                # Create a new mask, one for each label
                new_mask_compare = np.uint8((mask_copy_compare==label_indx)) # uint8 for the erode function
                # Find the edges of the mask
                kernel = np.ones((3, 3), np.uint8)
                img_erod_compare = cv2.erode(new_mask_compare, kernel)
                mask_edge_compare  = new_mask_compare - img_erod_compare
    
    
                # In order to draw the image, find each contour's coordinates.
                contours_compare, _ = cv2.findContours(mask_edge_compare, cv2.RETR_CCOMP , cv2.CHAIN_APPROX_SIMPLE)
    
                for cont_indx in range(len(contours_compare)):
                    vertices_compare = np.array(contours_compare[cont_indx],np.int32) # cv2 requires int32 points
                    pts_compare = vertices_compare.reshape(-1,1,2) # cv2 requires this format
    
    
                    cv2.polylines(new_img_3d_compare,[pts_compare],isClosed=True,
                                  color=np.multiply(to_rgb(my_colors[label_indx]),255),thickness=1) # Adds the polygon to the original image   
    
            ax[0].imshow(new_img_3d)          
            ax[0].legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0));
            ax[1].imshow(new_img_3d_compare)          
            ax[1].legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0)); 
            
    else: # If we don't need  to compare masks
        fig, ax = plt.subplots(figsize = fig_size); # We don't actually need "fig"
    
        # Plot the legend:
        # Since imshow can't really show a legend, we first plot something, and then overide it, while maintaining the legend
        for indx, color in enumerate(my_colors[:len(labels)]):
            ax.plot(2, 50, "-", color=color, label=labels[indx]);

    
        if title is not None:
            ax.set_title(title);
    
    

        if orig_img is None: # If we just want to plot the mask, no need for original image
            # Plot the image
            ax.imshow(mask,cmap=cmap, vmin=0, vmax=len(my_colors) - 1);          
            ax.legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0));
            
        else: # If we want to draw the mask on top of the original image
            ## Find and draw the edges
            # Make a copy to show the segmented image
            
            mask_copy = np.copy(mask)
    
            # Original image, copied
            if np.max(orig_img)>1.0:
                orig_img = orig_img/255
            new_img = np.copy(np.uint8(orig_img*255)) # uint 8 for the erode function
            # One less dimension (no need for a 1 dimension in the end)
            new_img = new_img[...,0]
            # Stack the same picture to get 3 color channels
            new_img_3d = np.stack((new_img,new_img,new_img),axis=-1)
    
            # Find the unique classes in the mask
            unique_val = np.delete(np.unique(mask_copy),[0]) # Delete the 0th index, we don't want to segment the background
            #print(tf.gather(tf.unique(tf.reshape(mask_copy,shape=-1))))
            
    
    
            #for label_indx in unique_val: # Find the mask for each unique label
            for label_indx in unique_val: # Find the mask for each unique label
                # Create a new mask, one for each label
                new_mask = np.uint8((mask_copy==label_indx)) # uint8 for the erode function
                # Find the edges of the mask
                kernel = np.ones((3, 3), np.uint8)
                img_erod = cv2.erode(new_mask, kernel)
                mask_edge  = new_mask - img_erod
    
    
                # In order to draw the image, find each contour's coordinates.
                contours, _ = cv2.findContours(mask_edge, cv2.RETR_CCOMP , cv2.CHAIN_APPROX_SIMPLE)
    
                for cont_indx in range(len(contours)):
                    vertices = np.array(contours[cont_indx],np.int32) # cv2 requires int32 points
                    pts = vertices.reshape(-1,1,2) # cv2 requires this format
    
    
                    cv2.polylines(new_img_3d,[pts],isClosed=True,color=np.multiply(to_rgb(my_colors[label_indx]),255),thickness=1) # Adds the polygon to the original image
    
            ax.imshow(new_img_3d)          
            ax.legend(labels,loc="upper left", bbox_to_anchor=(1.0, 1.0)); 

# Data Preprocessing

## Reading images and transforming them using a custom made image generator

In [ ]:
# We can add augmentations such as flips for the train and val sets
train_gen = data_gen(train_path_pics, train_path_mask, batch_size = batch_size, imsize=image_size,
                     n_classes = n_classes, seed = seed, augment = True, hist_eq = False)
                     #n_classes = n_classes, seed = seed, flip_up_down = False, flip_left_right = False, hist_eq = False)

val_gen = data_gen(val_path_pics, val_path_mask, batch_size = batch_size, imsize=image_size,
                   n_classes = n_classes, seed = seed, augment = False, hist_eq = False)
                   #n_classes = n_classes, seed = seed, flip_up_down = False, flip_left_right = False, hist_eq = False)

# We don't flip the test set, no augmentations
test_gen = data_gen(test_path_pics, test_path_mask, batch_size = batch_size, imsize=image_size,                    
                    n_classes = n_classes, seed = seed, augment = False, hist_eq = False)
                    #n_classes = n_classes, seed = seed, flip_up_down = False, flip_left_right = False, hist_eq = False)

In [ ]:
# Example
img,mask= train_gen.__next__()
for indx in range(img.shape[0]):# Show all the images
    #plt.figure()
    #plt.imshow(img[indx],cmap='gray')
    plt.figure();
    imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, orig_img = img[indx], title = 'Sgmentation mask', fig_size=(5, 5));
    if indx==3: # Show only 4 examples
        break

In [ ]:
# Histogram equalization - Option
img_show = img[0,...,0]
print(img_show.shape)
img_show = np.uint8(img_show*255)

fig, axs = plt.subplots(1, 2)
axs[0].imshow(img_show,cmap='gray')
axs[0].set_title('Normal')

img_eq = cv2.equalizeHist(img_show)
axs[1].imshow(img_eq,cmap='gray')
axs[1].set_title('Histogram equalization');

In [ ]:
print(mask.shape)
print(img.shape)

## Building the network

In [ ]:
# Classes:
# 'Background', 'Caries', 'Tooth filling', 'Secondary caries', 'Bone loss', 'Crown', 'Impacted tooth',
# 'Lesion RO','Lesion RL','Lesion Mixed','Implant'
weights_dict = {'Background':1, 'Caries':10, 'Tooth filling':1, 'Secondary caries':10, 'Bone loss':10, 'Crown':1,
                'Impacted tooth':1,'Lesion RO':1,'Lesion RL':1,'Lesion Mixed':1,'Implant':1}

weights_dict_serial = {}
for idx, key in enumerate(weights_dict):
    weights_dict_serial[idx] = weights_dict[key]

class_weight = np.array(weights_dict['Background'])
for label in labels:
    class_weight = np.append(class_weight, weights_dict[label])
    
#class_weight = list(class_weight/np.sum(class_weight))


#04/01/2025: For Caries, good weights: 30  | bad weights: 15

#class_weight = K.ones(n_classes,dtype=tf.float32)
#class_weight = np.ones(n_classes,dtype=np.float32)



print(f'class_weight = {class_weight}')
print(f'type(class_weight) = {type(class_weight)}')
print(f'np.sum(class_weight) = {np.sum(class_weight)}')
print(f'len(class_weight) = {len(class_weight)}')

In [ ]:
model = multi_unet_model(n_classes=n_classes, IMG_HEIGHT=image_size[0], IMG_WIDTH=image_size[1],
                         IMG_CHANNELS=image_size[2], seed=seed, lr = lr, class_weight = class_weight)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger, ModelCheckpoint, TensorBoard, BackupAndRestore, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=25)

import datetime
os.chdir(src_path)

if specific_annot:
    #prefix_name = 'Teeth_model_ Labels-'+ str(labels)
    prefix_name = 'Teeth_model_ Labels- partial (' + str(len(labels)) + ')'
else:
    prefix_name = 'Teeth_model_ Labels- all_labels'

time_name = '_ Time-' + datetime.datetime.now().strftime("%Y_%m_%d-%H_%M_%S")
#file_name = prefix_name +  'loss_func - {loss.__name__}' + '_ val_iou_score - {val_iou_score:.3f}' + time_name + '.keras'
file_name = prefix_name +  '_ val_iou_score - {val_iou_score:.3f}' + time_name + '.keras'

# Save the training data in a CSV file
csv_logger = CSVLogger(os.path.join(src_path,prefix_name + time_name + '.csv'))


# Save the model
model_checkpoint_callback = ModelCheckpoint(
    #filepath = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\checkpoint.model_{val_loss:.2f}.keras',
    filepath = os.path.join(src_path,file_name),
    monitor='val_loss',
    mode='min',
    #save_best_only=True,
    save_best_only=False,
    save_weights_only = False,
    save_freq="epoch")

backup_and_restore_callback = BackupAndRestore(
    backup_dir = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python'),
    #backup_dir = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python',
    save_freq="epoch", delete_checkpoint=True)

lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.8,
                              patience=5, verbose = 1, min_lr=1e-8, mode = 'min')

tensorboard_callback = TensorBoard(log_dir="./logs")
# C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python tensorboard --logdir logs
# https://www.tensorflow.org/tensorboard/get_started

In [ ]:
print(file_name)

In [ ]:
filepath = os.path.join(src_path,file_name)
print(filepath)

In [ ]:
# https://www.tensorflow.org/tutorials/images/segmentation?hl=en

In [ ]:
def display(display_list):
    plt.figure(figsize=(15, 15))

    title = ['Input Image', 'True Mask', 'Predicted Mask']

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow((display_list[i]),cmap='gray')
        plt.axis('off')
    plt.show()

In [ ]:
display([img[0],np.argmax(mask[0],-1)])

In [ ]:
def create_mask(pred_mask):
    pred_mask = tf.math.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0]

In [ ]:
def show_predictions(dataset=None):
    image, mask = dataset.__next__()
    pred_mask = model.predict(image)    
    imshow_mask_or_overlay(mask = mask[0], labels_list = labels, orig_img = image[0], title = 'L: Ground Truth | R: Prediction', fig_size = (10,10), mask_compare = pred_mask[0]);
    plt.show()

In [ ]:
show_predictions(train_gen)

In [ ]:
from IPython.display import clear_output
class DisplayCallback(tf.keras.callbacks.Callback):   
    def on_epoch_end(self, epoch, logs=None):
        #clear_output(wait=True)
        print ('\nSample Prediction after epoch {}\n'.format(epoch+1))
        show_predictions(train_gen)

# My custom function

In [ ]:
if not use_sm:
    #with tf.device('/CPU:0'):
    model.fit(x=train_gen, batch_size = batch_size, epochs=epochs, steps_per_epoch = num_images_train /  batch_size,
             verbose=1,
             validation_data = val_gen, validation_steps = num_images_val /  batch_size
              #,callbacks=[early_stop, csv_logger, model_checkpoint_callback, tensorboard_callback, DisplayCallback(train_gen)]
              ,callbacks=[early_stop,
                          #csv_logger, model_checkpoint_callback,
                          #tensorboard_callback, 
                          DisplayCallback()]
                         
    )

# Should I implement GracCam? It can help me see why the network decided to choose a certain class... 
## https://keras.io/examples/vision/grad_cam/

In [ ]:
# 1. Tell segmentation_models to use TensorFlow's internal Keras
os.environ["SM_FRAMEWORK"] = "tf.keras"

# 2. Monkey-patch the missing generic_utils attribute
tf.keras.utils.generic_utils = tf.keras.utils

# 3. In newer TensorFlow versions (Keras 3), get_custom_objects moved to keras.saving
if not hasattr(tf.keras.utils, 'get_custom_objects'):
    tf.keras.utils.get_custom_objects = tf.keras.saving.get_custom_objects

# 4. Safely import segmentation models
import segmentation_models as sm

import segmentation_models as sm

# Segmentation models library

In [ ]:
if use_sm:
    import segmentation_models as sm

    # For backbone, as of 04/02/2025, resnet34 is the best
    BACKBONE = 'resnet34'
    #BACKBONE = 'resnet152'
    #BACKBONE = 'mobilenetv2'
    #BACKBONE = 'inceptionv3'
    #BACKBONE = 'vgg19'

    # define model
    model = sm.Unet(BACKBONE, encoder_weights='imagenet', classes= n_classes, activation='softmax')

    dice_loss = sm.losses.DiceLoss()
    
    #dice_loss = sm.losses.DiceLoss(class_weights=class_weight)
    
    #dice_loss = sm.losses.DiceLoss(class_weights=class_weight, class_indexes= list(range(1,len(labels)+1)))
    
    focal_loss = sm.losses.CategoricalFocalLoss(alpha=0.25, gamma=2.0)#, class_indexes=None)# Worked as of 03/09/2025
    #focal_loss = sm.losses.CategoricalFocalLoss(alpha=0.25, gamma=5.0)

    # Doesn't work when we omit the Background class :(
    #focal_loss = sm.losses.CategoricalFocalLoss(alpha=0.25, gamma=2.0, class_indexes=list(range(1,len(labels)+1)))
    
    model.compile(
        optimizer = Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-07),
        #loss=dice_loss,
        loss=focal_loss,
        
        # class_indexes= list(range(1,len(labels)+1)) -> Do not count the Background in the calculation
        #metrics=[sm.metrics.IOUScore(class_weights=class_weight), sm.metrics.FScore(class_weights=class_weight)]
        
        metrics=[sm.metrics.IOUScore(class_indexes = list(range(1,len(labels)+1)), name = 'iou_score'),
                 sm.metrics.FScore(class_indexes = list(range(1,len(labels)+1)))]
    )

    # fit model
    model.fit(x=train_gen, batch_size = batch_size, epochs=epochs, steps_per_epoch = num_images_train //  batch_size,
             #verbose=1,
              verbose=2,
              #class_weight = weights_dict_serial,
             validation_data = val_gen, validation_steps = num_images_val //  batch_size
              ,callbacks=[early_stop,
                          csv_logger, model_checkpoint_callback,
                          backup_and_restore_callback,
                          lr_reduce,
                          #tensorboard_callback, 
                          DisplayCallback()])

In [ ]:
model_history = model.history.history
keys = model_history.keys()
final_results = {}
for key in keys:
    final_results[key] = model_history[key][-1]
print(final_results)

In [ ]:
# Keep track of our best results, from the last run
# Could be used to save the models and describe what the loss and metrics are.
model_history = model.history.history
keys = model_history.keys()
final_results = {}
for key in keys:
    final_results[key] = model_history[key][-1]
print(final_results)

In [ ]:
# Save the model
import datetime
os.chdir(src_path)

#if specific_annot:
#    file_name = 'Teeth_model_last_ Labels-'+ str(labels) + '_ val_iou_score - ' + str(np.round(final_results['val_iou_score'],2)) + '_ Time-' + datetime.datetime.now().strftime("%Y_%m_%d-%H_%M_%S") + '.keras'
#else:
#    file_name = 'Teeth_model_last_ Labels- all_labels' + '_ val_iou_score - ' + str(np.round(final_results['val_iou_score'],2)) + '_ Time-' + datetime.datetime.now().strftime("%Y_%m_%d-%H_%M_%S") + '.keras'

model.save(os.path.join(src_path, 'Last_model_' + prefix_name + time_name + '.csv'))


## Custom loss function & metric (They're already in the model file, it's here only if we want to load the model)

# Loss

# F1
def dice_coef_multilabel_loss_metric(loss=True,n_classes=2, class_weight= None):
    if class_weight is None:
        
        class_weight = K.ones(n_classes,dtype=tf.float32)
    class_weight_sum = K.cast(K.sum(class_weight), tf.float32)
    
    def dice_coef(y_true, y_pred, smooth=1e-6): 
        y_true_f = K.flatten(y_true)
        y_pred_f = K.flatten(y_pred)
        intersection = K.sum(y_true_f * y_pred_f)
        dice = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth) 
        return dice

    def dice_coef_multilabel(y_true, y_pred):
        dice = 0
        #dice_per_class = []
        for index in range(n_classes):
            dice += dice_coef(y_true[:,:,:,index], y_pred[:,:,:,index]) * class_weight[index]
            #dice += dice_coef(y_true[y_true[:,:,:,index]==1], y_pred[y_true[:,:,:,index]==1]) * class_weight[index]
            #dice_per_class.append(curr_dice.numpy())
        if loss:
            return -dice / class_weight_sum
        return dice / class_weight_sum
    return dice_coef_multilabel


# IOU
def jaccard_coef_loss_metric(loss=True, n_classes:int=2, class_weight= None):
    if class_weight is None:
        class_weight = K.ones(n_classes,dtype=tf.tf.float32)
    class_weight_sum = K.cast(K.sum(class_weight), tf.tf.float32)
        
    def jaccard_coef_single(y_true, y_pred, smooth=1e-7): # Stable, working for teeth segmentation
        y_true_f = K.cast(K.flatten(y_true),dtype=tf.tf.float32)
        y_pred_f = K.cast(K.flatten(y_pred),dtype=tf.tf.float32)
        intersection = K.sum(y_true_f * y_pred_f) + smooth
        #intersection = K.sum(y_true_f * y_pred_f)
        union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth
        return intersection / union
        #return (intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth)

    def jaccard_coef_multilabel(y_true, y_pred):
        iou = 0.0
        #iou_per_class = []
        num_of_ground_truth_labels = 0.0 # Checks how much ground truth labels were present 
        for index in range(n_classes):
            #iou += jaccard_coef_single(y_true[:,:,:,index], y_pred[:,:,:,index]) * class_weight[index]
            if not tf.equal(tf.math.count_nonzero(y_true[:,:,:,index]==1), 0): # Checks if there are no ground truth for a certine label
                iou += jaccard_coef_single(y_true[y_true[:,:,:,index]==1], y_pred[y_true[:,:,:,index]==1]) * class_weight[index]
                num_of_ground_truth_labels+=1
            #iou_per_class.append(curr_iou.numpy())
            
        if loss:
            return -iou / num_of_ground_truth_labels
        return iou / num_of_ground_truth_labels
    return jaccard_coef_multilabel

# Metrics

# F1
def dice_coef_multilabel(y_true, y_pred, n_classes, class_weight= None):
    
    if class_weight is None:
        class_weight = K.ones(n_classes,dtype = tf.float32)
    class_weight_sum = K.cast(K.sum(class_weight), tf.float32)
    
    def dice_coef(y_true, y_pred, smooth=1e-6): 
        y_true_f = K.flatten(y_true)
        y_pred_f = K.flatten(y_pred)
        intersection = K.sum(y_true_f * y_pred_f)
        dice = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth) 
        return dice

    dice = 0
    dice_per_class = []
    for index in range(n_classes):
        #dice += dice_coef(y_true[:,:,:,index], y_pred[:,:,:,index]) * class_weight[index]
        curr_dice = dice_coef(y_true[y_true[:,:,:,index]==1], y_pred[y_true[:,:,:,index]==1]) * class_weight[index]
        dice += curr_dice
        dice_per_class.append(curr_dice.numpy())
    return dice / class_weight_sum

# IOU
def jaccard_coef_multilabel(y_true, y_pred, smooth=1e-6,loss=True, n_classes:int=2, class_weight= None):
    if class_weight is None:
        class_weight = K.ones(n_classes,dtype=tf.float32)
    class_weight_sum = K.cast(K.sum(class_weight), tf.float32)
        
    def jaccard_coef_single(y_true, y_pred, smooth=1e-7): # Stable, working for teeth segmentation
        y_true_f = K.cast(K.flatten(y_true),dtype=tf.float32)
        y_pred_f = K.cast(K.flatten(y_pred),dtype=tf.float32)
        intersection = K.sum(y_true_f * y_pred_f) + smooth
        #intersection = K.sum(y_true_f * y_pred_f)
        union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth
        return intersection / union
        #return (intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth)

    iou = 0.0
    #iou_per_class = []
    num_of_ground_truth_labels = 0.0 # Checks how much ground truth labels were present 
    for index in range(n_classes):
        #iou += jaccard_coef_single(y_true[:,:,:,index], y_pred[:,:,:,index]) * class_weight[index]
        if not tf.equal(tf.math.count_nonzero(y_true[:,:,:,index]==1), 0): # Checks if there are no ground truth for a certine label
            iou += jaccard_coef_single(y_true[y_true[:,:,:,index]==1], y_pred[y_true[:,:,:,index]==1]) * class_weight[index]
            num_of_ground_truth_labels+=1
        #iou_per_class.append(curr_iou.numpy())

    return iou / num_of_ground_truth_labels

'''def jaccard_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) - intersection + smooth)'''

## Find the f1 score and IOU of the test set

test_gen_metrics = data_gen(test_path_pics, test_path_mask, batch_size = num_images_test, imsize=image_size, n_classes = n_classes,
                            seed = seed, flip_up_down=False, flip_left_right=False)

# Get all of the images
img,mask= test_gen_metrics.__next__()
test_f1_score = dice_coef_multilabel(mask, model.predict(img), n_classes=n_classes, class_weight = class_weight).numpy()
test_IOU_score = jaccard_coef_multilabel(mask, model.predict(img), class_weight = class_weight).numpy()

train_f1_score = final_results['dice_coef_multilabel']
train_IOU_score = final_results['jaccard_coef_multilabel']

val_f1_score = final_results['val_dice_coef_multilabel']
val_IOU_score = final_results['val_jaccard_coef_multilabel']


print(f'The f1 score of the train set is {100*train_f1_score:.2f}%')
print(f'The IOU score of the train set is {100*train_IOU_score:.2f}%')
print(f'The f1 score of the val set is {100*val_f1_score:.2f}%')
print(f'The IOU score of the val set is {100*val_IOU_score:.2f}%')
print(f'The f1 score of the test set is {100*test_f1_score:.2f}%')
print(f'The IOU score of the test set is {100*test_IOU_score:.2f}%')

In [ ]:
# Optional - load the model!
# Since we used custom loss and metrics, we first load our model, them we compile it again
# (we HAVE to define the loss and metric functions again here)


'''
#os.chdir(r'C:\\Users\\aviel\\Documents\\BIU - 2nd Degree\\Thesis\\Image_segmentation_python\\Best models')
os.chdir(r'C:\\Users\\aviel\\Documents\\BIU - 2nd Degree\\Thesis\\Image_segmentation_python')
model_saved = tf.keras.models.load_model("checkpoint.model.keras", compile=False)

model_saved.compile(optimizer = 'adam', loss = [dice_coef_multilabel_loss_metric(loss = True,n_classes = n_classes)],
                  metrics = [dice_coef_multilabel_loss_metric(loss = False, n_classes = n_classes),
                             #jaccard_coef_multilabel(loss = False, n_classes = n_classes),
                             'categorical_accuracy'])
model = model_saved
'''


#### BESST OPTION


import segmentation_models as sm
os.chdir(os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python'))
dice_loss = sm.losses.DiceLoss() 
focal_loss = sm.losses.CategoricalFocalLoss(alpha=0.25, gamma=2.0)#, class_indexes=None)
iou_score = sm.metrics.IOUScore(class_indexes = list(range(1,len(labels)+1)), name = 'iou_score')
f1_score = sm.metrics.FScore(class_indexes = list(range(1,len(labels)+1)))

file_name = (os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/Best models')
#file_name = os.path.join(file_name,'Teeth_model_ Focal_Loss_Labels- all_labels_ val_iou_score - 0.23_ Time-2025_12_01-07_40_02.keras')

# Dice Loss
'''
file_name = os.path.join(file_name,'Teeth_model_ Focal_loss_Labels- all_labels_ val_iou_score - 0.575_ Time-2026_01_17-18_30_43.keras')

model_saved_dice = tf.keras.models.load_model(file_name, compile=True,
                                        custom_objects={'dice_loss': dice_loss, 'iou_score': iou_score, 'f1-score': f1_score})

model = model_saved_dice
'''

# Focal Loss

#'''
#file_name = os.path.join(file_name,'Teeth_model_ Focal_Loss_Labels- all_labels_ val_iou_score - 0.252_ Time-2026_01_16-09_28_09.keras')
file_name = os.path.join(src_path,'Teeth_model_ Labels- partial (2)_ val_iou_score - 0.056_ Time-2026_01_25-15_07_17.keras')

model_saved_focal = tf.keras.models.load_model(file_name, compile=True,
                                        custom_objects={'focal_loss': focal_loss, 'iou_score': iou_score, 'f1-score': f1_score})
model = model_saved_focal
#'''
                                        
#### BESST OPTION


'''
model_saved = tf.keras.models.load_model("Teeth_model_last_ Labels- all_labels_ iou score - 0.31_ Time-2025_08_20-22_23_48.keras", compile=False)
model_saved.compile(
        optimizer = 'Adam',
        #loss=dice_loss,
        loss=focal_loss,
        #loss=binary_focal_loss,
        
        # class_indexes= list(range(1,len(labels)+1)) -> Do not count the Background in the calculation
        #metrics=[sm.metrics.IOUScore(class_indexes= list(range(1,len(labels)+1))),
        #         sm.metrics.FScore(class_indexes= list(range(1,len(labels)+1)))]
        #metrics=[sm.metrics.IOUScore(class_weights=class_weight), sm.metrics.FScore(class_weights=class_weight)]
        metrics=[sm.metrics.IOUScore(class_indexes = list(range(1,len(labels)+1)), name = 'iou_score'),
                 sm.metrics.FScore()])
model = model_saved
'''

In [ ]:
# If we want, we can train again
'''
model_saved.fit(x=train_gen, batch_size = batch_size, epochs=epochs, steps_per_epoch = num_images_train /  batch_size,
         verbose=1
          , validation_data = val_gen, validation_steps = num_images_val /  batch_size,
        # callbacks=[early_stop]
         )'''

## Function: Calculate the metrics for *ALL DATASET*

In [ ]:
import pandas as pd

def find_metrics(generator, num_images, model, train_val_test=''):
    tot_step = int(np.ceil(num_images /  batch_size))
    cm = 0
    eps = np.finfo(float).eps
    for indx in range(tot_step):
        img,mask= generator.__next__()
        mask_pred = model.call(inputs = tf.constant(img))
        cm = cm + confusion_matrix(y_true =  K.flatten(K.argmax(mask,axis=-1)),
                                   y_pred = K.flatten(K.argmax(mask_pred,axis=-1)), labels = np.arange(0,n_classes))

        # Calculate metrics for all train images
    precision_list=[]
    recall_list=[]
    accuracy_list=[]
    iou_list=[]
    f1_list=[]
    for indx in range(n_classes):

        # We're adding epsilon for each one to avoide dvision by 0
        # True positive
        tp = cm[indx,indx]
        # True negative
        tn = np.sum(cm[np.concatenate((np.arange(0,indx),np.arange(indx+1,n_classes-1))),
        np.concatenate((np.arange(0,indx),np.arange(indx+1,n_classes-1)))])
        # False positive
        fp = np.sum(cm[indx,np.concatenate((np.arange(0,indx),np.arange(indx+1,n_classes-1)))])
        # False negative
        fn = np.sum(cm[np.concatenate((np.arange(0,indx),np.arange(indx+1,n_classes-1))),indx])  
        # We can also add an epsilon to the numerator
        precision_list.append((tp)/(tp+fp+eps))
        recall_list.append((tp)/(tp+fn+eps))
        accuracy_list.append((tp+tn)/(tp+fp+fn+tn+eps))
        iou_list.append((tp) / (tp + fp + fn+eps))
        f1_list.append((2*tp)/(2*tp+fp+fn + eps))
    df = pd.DataFrame(data=np.stack((precision_list, recall_list, accuracy_list, iou_list, f1_list),axis=0),
     index=['precision','recall','accuracy', 'iou', 'f1 / Dice'],columns = ['Background'] + labels)
    
    path_name = os.path.join(src_path,'All metrics '+ train_val_test + ' - ' +  datetime.datetime.now().strftime("%Y_%m_%d-%H_%M_%S") +'.xlsx')
    df.to_excel(path_name,index=True)
    return df

## Check the training set

In [ ]:
metrics_train = find_metrics(train_gen, num_images = num_images_train, model = model, train_val_test = 'train')

In [ ]:
metrics_train.head()

In [ ]:
img,mask= train_gen.__next__()
print(img.shape)

mask_pred = model.predict(img)
#K.max(mask_pred,axis=-1)
#K.argmax(mask_pred,axis=-1)

# X axis is the actual label (i.e, every pixel in r 0 row is actually bacground pixel)
# and the Y axis is the predicted label (i.e column 0 predicts that the pixel is background)

# Classification report
print(classification_report(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1)),
                            labels = np.arange(0,n_classes), target_names = ['Background'] + labels, zero_division =0))


# Confusion matrix
print(confusion_matrix(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1))))

#print(tf.math.confusion_matrix(labels=K.flatten(K.argmax(mask,axis=-1)), predictions = K.flatten(K.argmax(mask_pred,axis=-1)), num_classes=n_classes))

In [ ]:
for indx in range(len(img)):
    #plt.figure()
    #plt.imshow(img[indx],cmap='gray')
    #plt.title('Original image')
    plt.figure();
    mask_pred = model.call(inputs = tf.constant(img))
    imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, orig_img = img[indx], title = 'Sgmentation mask', mask_compare = mask_pred[indx]);
    
    if indx==7: # Show only 7 images
        break;

## Check the validation set

In [ ]:
metrics_val = find_metrics(val_gen, num_images = num_images_val, model = model, train_val_test = 'val')

In [ ]:
metrics_val.head()

In [ ]:
img,mask= val_gen.__next__()
print(img.shape)
#mask_pred = model.predict(img)

# Classification report
print(classification_report(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1)),
                            labels = np.arange(0,n_classes), target_names = ['Background'] + labels, zero_division =0))
# Confusion matrix
print(confusion_matrix(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1))))

#print(tf.math.confusion_matrix(labels=K.flatten(K.argmax(mask,axis=-1)), predictions = K.flatten(K.argmax(mask_pred,axis=-1)), num_classes=n_classes))

In [ ]:
for indx in range(len(img)):
    #plt.figure()
    #plt.imshow(img[indx],cmap='gray')
    #plt.title('Original image')
    plt.figure();
    #plt.imshow(np.argmax(image,axis=-1),cmap='gray',vmin=0,vmax=n_classes-1)
    #imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, title = 'Original mask')
    mask_pred = model.call(inputs = tf.constant(img))
    imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, orig_img = img[indx], title = 'Sgmentation mask', mask_compare = mask_pred[indx]);
    
    if indx==7: # Show only 7 images
        break;

In [ ]:
## Check the test set

In [ ]:
metrics_test = find_metrics(test_gen,num_images = num_images_test, model = model, train_val_test = 'test')

In [ ]:
metrics_test.head()

In [ ]:
## Overlay the predicted images on the real images
# Get a batch of images
img,mask= test_gen.__next__()
print(img.shape)
print(mask.shape)
#mask_pred = model.predict(img)

# Classification report
print(classification_report(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1)),
                            labels = np.arange(0,n_classes), target_names = ['Background'] + labels, zero_division =0))
# Confusion matrix
print(confusion_matrix(y_true =  K.flatten(K.argmax(mask,axis=-1)), y_pred = K.flatten(K.argmax(mask_pred,axis=-1))))

#print(tf.math.confusion_matrix(labels=K.flatten(K.argmax(mask,axis=-1)), predictions = K.flatten(K.argmax(mask_pred,axis=-1)), num_classes=n_classes))

In [ ]:
for indx in range(len(img)):
    #plt.figure()
    #plt.imshow(img[indx],cmap='gray')
    #plt.title('Original image')
    plt.figure();
    #plt.imshow(np.argmax(image,axis=-1),cmap='gray',vmin=0,vmax=n_classes-1)
    #imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, title = 'Original mask')
    mask_pred = model.call(inputs = tf.constant(img))
    imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, orig_img = img[indx], title = 'Sgmentation mask', mask_compare = mask_pred[indx]);
    
    if indx==7: # Show only 7 images
        break;

## Meeting with dentists 15/09/2025

In [ ]:
#test_2_path = r'C:\Users\aviel\Documents\Dental_ai\Mishel'
#test_2_path = r'C:\Users\aviel\Documents\Dental_ai\Karin'
#test_2_path = r'C:\Users\aviel\Documents\Dental_ai\Biana'
#test_2_path = r'C:\Users\aviel\Documents\Dental_ai\Egor'


imgs = []
file_names = []
for root, dirs, files in os.walk(top = test_2_path):
    for image in files:
        picc = cv2.imread(os.path.join(root,image))/255.0
        picc =  cv2.resize(picc, (image_size[0], image_size[1]), interpolation = cv2.INTER_NEAREST)
        imgs.append(picc)
        file_names.append(image)
        
        #picc2 = np.expand_dims(picc, axis=0)
        #masks_pred.append(model.call(inputs = tf.constant(picc2)))

#masks_pred = np.array(masks_pred, dtype=np.uint8)

In [ ]:
masks_pred = model.call(inputs = tf.constant(imgs))

In [ ]:
curr_indx = 0

In [ ]:
for count,indx in enumerate(range(curr_indx,len(imgs))):
    plt.figure()
   
    #imshow_mask_or_overlay(mask = mask_pred_test, labels_list = labels, title = 'Predicted mask')
    imshow_mask_or_overlay(mask = masks_pred[indx], labels_list = labels, orig_img = imgs[indx], title = file_names[indx]);
    curr_indx = indx
    if count==3: # Show only 4 images
        curr_indx = curr_indx+1
        break;


In [ ]:
for indx in range(len(img)):
    plt.figure()
    plt.imshow(img[indx],cmap='gray')
    plt.title('Original image')
    plt.figure();
    #plt.imshow(np.argmax(image,axis=-1),cmap='gray',vmin=0,vmax=n_classes-1)
    #imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, title = 'Original mask')
    imshow_mask_or_overlay(mask = mask[indx], labels_list = labels, orig_img = img[indx], title = 'Image sgmentation - original mask');

    plt.figure()
    mask_pred = model.call(inputs = tf.constant(img))
    mask_pred_test = mask_pred[indx]
    
    #imshow_mask_or_overlay(mask = mask_pred_test, labels_list = labels, title = 'Predicted mask')
    imshow_mask_or_overlay(mask = mask_pred_test, labels_list = labels, orig_img = img[indx], title = 'Image sgmentation - predicted mask');
    
    if indx==7: # Show only 7 images
        break;


In [ ]:
def to_categorical(x, num_classes=None):
    """Converts a class vector (integers) to binary class matrix.

    E.g. for use with `categorical_crossentropy`.

    Args:
        x: Array-like with class values to be converted into a matrix
            (integers from 0 to `num_classes - 1`).
        num_classes: Total number of classes. If `None`, this would be inferred
            as `max(x) + 1`. Defaults to `None`.

    Returns:
        A binary matrix representation of the input as a NumPy array. The class
        axis is placed last.
    """

    x = np.array(x, dtype="int64")
    input_shape = x.shape
    # Shrink the last dimension if the shape is (..., 1).
    if input_shape and input_shape[-1] == 1 and len(input_shape) > 1:
        input_shape = tuple(input_shape[:-1])

    x = x.reshape(-1)
    if not num_classes:
        num_classes = np.max(x) + 1
    batch_size = x.shape[0]
    categorical = np.zeros((batch_size, num_classes))
    categorical[np.arange(batch_size), x] = 1
    output_shape = input_shape + (num_classes,)
    categorical = np.reshape(categorical, output_shape)
    return categorical

In [ ]:
# Interesting images from test set: 190 (Tooth filling, Bones loss, Lesion RO, Lesion RL), 409 (Crown, Implant),
# 3659 (Secondary Caries, bone loss, crown), 3805 (Caries, Tooth filling, Bone loss, Lesion RL), 4649 (Bone loss, Impacted tooth)
# 5422 (Caries, Tooth filling, Crown, Lesion Mixed, Implant)

file_name = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/test/test_pics/5422.tif')
#file_name = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\test\test_pics\5422.tif'
mask_name = os.path.join(prefix_os_c,'Users/aviel/Documents/BIU - 2nd Degree/Thesis/Image_segmentation_python/My_pics/test/test_masks/5422.png')
#mask_name = r'C:\Users\aviel\Documents\BIU - 2nd Degree\Thesis\Image_segmentation_python\My_pics\test\test_masks\5422.png'

mask_img =cv2.imread(mask_name,0)
mask_img = cv2.resize(mask_img, (image_size[1], image_size[0]), interpolation = cv2.INTER_NEAREST)
mask_img = to_categorical(mask_img,n_classes)
#print(mask_img.shape)

picc = cv2.imread(file_name)/255.0
picc =  cv2.resize(picc, (image_size[0], image_size[1]), interpolation = cv2.INTER_NEAREST)
#print(picc.shape)|
picc2 = np.expand_dims(picc, axis=0)
#print(picc2.shape)
#plt.imshow(picc2[0])
mask_pred = model.predict(picc2)
#mask_pred.shape
imshow_mask_or_overlay(mask = mask_img, labels_list = labels, orig_img = picc, title = 'Image sgmentation - Prediction mask', mask_compare=mask_pred[0]);

In [ ]:
x123 = mask_pred[0]

In [ ]:
x123[310,220]

In [ ]:
plt.imshow(np.argmax(mask_pred[0],axis=-1)[355:395,365:402])

mask_pred = model.predict(img)

plt.figure()
plt.imshow(mask[0,:,:,0],cmap='gray')
plt.figure()
plt.imshow(mask_pred[0,:,:,0],cmap='gray')
print(mask_pred[0,200,20])

In [ ]:
'''# Release the GPU memory
device = cuda.get_current_device()
print(device)
device.reset()'''